## Procesamiento y sistematización de extractos bancario

Esta notebook surge de la necesidad de transformar información bancaria histórica, almacenada en múltiples extractos mensuales en formato PDF, en datos estructurados y analizables.

El proceso consiste en:

- Extraer los movimientos diarios de cada extracto bancario.
- Transformar y limpiar la información obtenida.
- Consolidar todos los períodos en un único DataFrame.

La información final incluye la fecha, número de comprobante, concepto, débitos, créditos y saldo de la cuenta corriente, y queda disponible para su posterior análisis y actualización.


In [71]:
import pdfplumber
import pandas as pd
import re # Permite trabajar con expresiones regulares
import os
print("Librerias ok")

Librerias ok


### Prueba unitaria con un solo archivo

In [ ]:
# Ruta de un solo archivo para empezar a probar
ruta_pdf = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/resumenes_bancarios/Banco_05-2026.pdf"


In [ ]:
# Primer prueba del procesamiento del arhico del resumen bancario
# 1. Columnas finales
# Nombres en el archivo
columnas = ["Fecha", "Comprobante", "Movimiento", "Débito", "Crédito", "Saldo en cuenta"]

# 2. Patron de fechas con regex
patron_fecha = re.compile(r"\b\d{2}/\d{2}/\d{2}\b") # .compile prepara un patrón de búsqueda.

# 3. Distintos patrones de montos
# Por ejemplo: $ 24.543,75 // -$ 3.051.185,33 // $ 33,87 // $ 800.000,00 /
patron_monto = re.compile(r"-?\$\s?"r"\d{1,3}"r"(?:\.\d{3})*"r",\d{2}") 

# 4. Convertir monto de texto a numero 
# porque en Arg por ejemplo los decimales vienen con , y necestamos que sean con .
# si no hay valor fila-fila se pone NA. Tambien considera los numeros negativos
def convertir_monto(texto):
    if texto is None:
        return "NA"
    texto = texto.strip()
    negativo = "-" in texto
    texto = (texto.replace("$", "").replace("-", "").replace(" ", "").replace(".", "").replace(",", "."))
    try:
        valor = float(texto)
    except ValueError:
        return "NA"
    if negativo:
        valor *= -1
    return valor

# 5. Configuracion de la fecha en el inicio de la linea
# Hay veces que aparecen fechas en el concepto pero eso no lo tenemos que tner en cuenta
def fecha_al_inicio(texto):
    resultado = re.match(r"^\s*(\d{2}/\d{2}/\d{2})", texto)
    if resultado:
        return resultado.group(1)
    return None


# 6. Agrupar palabras en linea
def agrupar_palabras_en_lineas(palabras, tolerancia_vertical = 3):
    palabras = sorted(palabras, key = lambda palabra: palabra["top"])
    lineas = []
    for palabra in palabras:
        agregada = False
        for linea in lineas:
            top_linea = linea[0]["top"]
            if (abs(palabra["top"] - top_linea) <= tolerancia_vertical):
                linea.append(palabra)
                agregada = True
                break
        if not agregada:
            lineas.append([palabra])
    # Ordenar horizontalmente
    for linea in lineas:
        linea.sort(key=lambda palabra: palabra["x0"])
    return lineas

# 7. Extraer solo fecha
def es_fecha_sola(texto):
    #print(texto)
    return bool(re.fullmatch(r"\d{2}/\d{2}/\d{2}", texto.strip()))

# 8. Setear el numero de pagina 
def es_numero_de_pagina(texto):
    return bool(re.fullmatch(r"\d+\s*-\s*\d+", texto.strip()))

# 9. Informacion que no interesa 
#Otras para agregar?? es la info que esta al final o al principio del pdf
def es_informacion_a_ignorar(texto):
    texto_lower = texto.lower()
    # Informacion de la cta bancaria
    if "cuenta corriente nº" in texto_lower:
        return True
    if "cuenta corriente nro" in texto_lower:
        return True
    if "cbu:" in texto_lower:
        return True
    # Informacion del banco
    if ("banco santander argentina s.a." in texto_lower):
        return True
    if ("correlativo 800678" in texto_lower):
        return True 
    if ("ningún accionista mayoritario" in texto_lower):
        return True
    if ("tampoco lo hacen otras entidades" in texto_lower):
        return True
    if ("salvo error u omisión" in texto_lower):
        return True
    
    # Informacion del acuerdo bancario
    if texto_lower == "acuerdo":
        return True
    if texto_lower.startswith("límite:"):
        return True
    if texto_lower.startswith("limite:"):
        return True
    if "vencimiento:" in texto_lower:
        return True
    if "total numerales:" in texto_lower:
        return True
    if "total excedido:" in texto_lower:
        return True
    if "máximo saldo deudor:" in texto_lower:
        return True
    if "maximo saldo deudor:" in texto_lower:
        return True

    # Encabezado de la tabla 
    if ("fecha" in texto_lower  and "comprobante" in texto_lower and "movimiento" in texto_lower):
        return True
    return False

# 10. Lineas explicativas
# revisar pra otros resumenes bancarios si hay otras palabras
def es_linea_explicativa(texto):
    texto_lower = (texto.lower().strip())
    patrones = ["resp:", "a ", "de ", "del ", "comision transferencias", "comisión transferencias", "lx argentina",
        "federacion patro", "federación patro", "bbva consolidar", "sancor cooperati", "camino de las si"]
    for patron in patrones:
        if texto_lower.startswith(patron):
            return True
    return False

# 11. Limpiar info del movimiento
def limpiar_movimiento(texto):
    # Eliminar solamente una fecha
    # si aparece al inicio del texto
    texto = re.sub(r"^\s*\d{2}/\d{2}/\d{2}\s*", "", texto)
    # Eliminar espacios repetidos
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

# 12. Extraer el numero de comprobante y el movimiento
def extraer_comprobante_y_movimiento(texto):
    # Eliminar fecha solamente al principio
    texto = re.sub(r"^\s*\d{2}/\d{2}/\d{2}\s*", "", texto)

    # Eliminar montos
    texto = patron_monto.sub("", texto)

    # Limpiar espacios
    texto = re.sub(r"\s+"," ", texto).strip()

    # Separar comprobante
    partes = texto.split()
    comprobante = "NA"
    if (len(partes) > 0 and partes[0].isdigit()):
        comprobante = partes[0]
        partes = partes[1:]
    movimiento = " ".join(partes)
    movimiento = limpiar_movimiento(movimiento)
    return comprobante, movimiento

# 13. Variables principales
registros = []
leyendo_movimientos = False
fin_movimientos = False
fecha_actual = None
ultimo_registro = None
# Variable para guardar el saldo total
# que aparece en la ultima línea despues del ultimo valor de la tabla
saldo_total = None

# 14. Recorrer todo el pdf
with pdfplumber.open(ruta_pdf) as pdf:
    print(f"Cantidad de páginas: " f"{len(pdf.pages)}")
    for numero_pagina, pagina in enumerate(pdf.pages, start=1):
        # Si terminó la tabla, no procesar más páginas
        if fin_movimientos:
            break
        print(f"Procesando página " f"{numero_pagina}...")

        # Extraer palabras
        palabras = pagina.extract_words()
        # Agrupar palabras en líneas
        lineas = agrupar_palabras_en_lineas(palabras)

        # Procesar cada línea
        for linea in lineas:
            texto = " ".join(palabra["text"] for palabra in linea)
            texto = texto.strip()
            # Inicio de movimientos
            if (texto.lower() == "movimientos"):
                leyendo_movimientos = True
                print("Inicio de movimientos")
                continue
            # Fin de procemiento por saldo total
            #Esto se agrego despues porque necesaitaba este valor para hacer una comprobacion de que este todo ok, debitos +creditos = saldo
            if (leyendo_movimientos and "saldo total" in texto.lower()):
                print( "Fin movimientos: " "Saldo total encontrado")

                # Extraer el monto del saldo total
                montos_saldo_total = (patron_monto.findall(texto))
                if len(montos_saldo_total) > 0: 
                    saldo_total = convertir_monto(montos_saldo_total[-1])
                fin_movimientos = True
                leyendo_movimientos = False
                break #Salir

            # Fin por detalle impositivo, que seria la seccion que viene despues
            # Esto es porque se conoce el pdf bancario
            if (leyendo_movimientos and "detalle impositivo" in texto.lower()):
                print("Fin de movimientos")
                fin_movimientos = True
                leyendo_movimientos = False
                break

            # Si no estamoos en movimientos
            # Creo que esta parte se repite porque ya se define antes, revisar!
            if not leyendo_movimientos:
                continue

            # Ignorar informacion general delp pdf
            if es_informacion_a_ignorar(texto):
                continue
            # Ignorar el numero de pagina
            if es_numero_de_pagina(texto):
                continue

            # Actualizar la info de la fecha             #
            # SOLO si está al principio, esto evita que fechas internas como "Del 01/04/26 al 30/04/26"
            # cambien la fecha del movimiento.
            fecha_en_linea = fecha_al_inicio(texto)
            if fecha_en_linea:
                fecha_actual = fecha_en_linea

            # Extraer montos
            montos = patron_monto.findall(texto)
            
            # Lineas explicativas importantes 
            if es_linea_explicativa(texto):
                if ultimo_registro is not None:
                    texto_limpio = limpiar_movimiento(texto)
                    if texto_limpio:
                        ultimo_registro["Movimiento"] += (" " + texto_limpio)
                continue

            # Lineas sin montos 
            if len(montos) == 0:
                # Fecha sola
                if es_fecha_sola(texto):
                    continue
                # Texto explicativo
                if ultimo_registro is not None:
                    texto_limpio = limpiar_movimiento(texto)
                    if texto_limpio:
                        ultimo_registro["Movimiento"] += (" " + texto_limpio)
                continue

            # Ultimo monto = saldo, esto se repite creo? revisar
            saldo = convertir_monto(montos[-1])

           # El penunltimo valor corresponde al Importe final del mes 
            if len(montos) >= 2:
                importe = convertir_monto(montos[-2])
            else:
                importe = "NA"

            # Extraer comprobante y movimiento

            texto_sin_montos = patron_monto.sub("", texto)
            comprobante, movimiento = (extraer_comprobante_y_movimiento(texto_sin_montos))

            # Hay que ver, clasificar si corresponde a valor de debito o credito
            #Una forma: Se compara el saldo actual con el saldo anterior.
            # # Si el saldo aumenta: Credito
            # Si el saldo disminuye: Debito
            
            #Otra forma: con los espacios pero no funciono porque el monto puede variar y la cantidad de espacios/caracteres puede variar
            debito = "NA"
            credito = "NA"
            #Verifica que no este vacio, que sea entero o decimal, 
            if (len(registros) > 0 and isinstance(saldo,(int, float)) and isinstance(registros[-1]["Saldo en cuenta"],(int, float))
                and isinstance(importe,(int, float))):
                saldo_anterior = (registros[-1]["Saldo en cuenta"])
                # Saldo aument
                if saldo > saldo_anterior:
                    credito = importe
                # Saldo disminuye
                elif saldo < saldo_anterior:
                    debito = importe
            else:
                # Primer movimiento: Saldo Inicial que no tiene débito ni crédito.
                # Sirve para validar la info al final
                debito = "NA"
                credito = "NA"

            # Crear registro
            registro = {"Fecha": fecha_actual, "Comprobante": comprobante,"Movimiento": movimiento, "Débito": debito,"Crédito": credito,
                "Saldo en cuenta": saldo}
            registros.append(registro)
            ultimo_registro = registros[-1]


# 15. Crear df
df_movimientos = pd.DataFrame(registros, columns = columnas)

# 16. Completar info vacio
df_movimientos = (df_movimientos.fillna("NA"))

# 17. Guardar saldo inicial
saldo_inicial = (df_movimientos.iloc[0]["Saldo en cuenta"])

# 18. Eliminar saldo inicial
df_movimientos = (df_movimientos.iloc[1:].reset_index(drop=True))

# 19. Limpieza final del movimiento de interes
df_movimientos["Movimiento"] = (df_movimientos["Movimiento"].astype(str).str.replace(r"\s+"," ",regex=True).str.strip())
#print(df_movimientos["Movimiento"] )

# 20. Convertir debitos y creditos a numeros
df_movimientos["Débito_num"] = pd.to_numeric(df_movimientos["Débito"],errors="coerce") # que es coerce? lo dejamos prque asi funciona
df_movimientos["Crédito_num"] = pd.to_numeric(df_movimientos["Crédito"], errors="coerce")


# 21. Calcular total de debitos
total_debitos = (df_movimientos["Débito_num"].fillna(0).sum())
#print(total_debitos)

# 22. Calcular creditos
total_creditos = (df_movimientos["Crédito_num"].fillna(0).sum())

# 23. Calcular el saldo final para comprobar que todos los debitos/creditos estan bien ubicados
# y que no hay valores de mas
saldo_calculado = (saldo_inicial - total_debitos + total_creditos)


# 24. CEvaluacion final si est todo ok 
# print("\n")
# print("=" * 100)
# print("Comprobacion de saldo")
# print("=" * 100)
# print("Saldo inicial:", saldo_inicial)
# print("Total débitos:", total_debitos)
# print( "Total créditos:", total_creditos)
# print("Saldo calculado:",saldo_calculado)
# print("Saldo total del extracto:",saldo_total)
# print("\n")

# Comparar 
if (round(saldo_calculado, 2) == round(saldo_total, 2)):
    print("Comprobacion correcta")
    print("El saldo inicial, los débitos y los créditos coinciden con el saldo total del extracto.")
else: 
    diferencia = (saldo_calculado -saldo_total)
    print("Comprobacion incorrecta")
    print("Diferencia:", round(diferencia, 2))

# 25. Eliminar algunas columnas que no son de interes
df_movimientos = df_movimientos.drop(columns=["Débito_num","Crédito_num"])

# 26. Mostrar info del df final
print("\n")
print("=" * 100)
print("DATAFRAME FINAL")
print("=" * 100)
# print(df_movimientos)
print("\n")
print("Cantidad total de movimientos:", len(df_movimientos))


# # 26. Exportar excel/csv
# df_movimientos.to_excel("movimientos_bancarios.xlsx", index=False)
# df_movimientos.to_csv("movimientos_bancarios.csv")


Cantidad de páginas: 9
Procesando página 1...
Inicio de movimientos
Procesando página 2...
Procesando página 3...
Procesando página 4...
Procesando página 5...
Procesando página 6...
Fin movimientos: Saldo total encontrado
Comprobacion correcta
El saldo inicial, los débitos y los créditos coinciden con el saldo total del extracto.


DATAFRAME FINAL


Cantidad total de movimientos: 126


In [ ]:
#Lo guardamos para revisar la prueba unitaria
df_movimientos.to_csv("prueba.csv")

### Contruccion de funcion completa

In [ ]:
def extraer_movimientos_bancarios(ruta_pdf):
    registros = []

    leyendo_movimientos = False
    fin_movimientos = False
    fecha_actual = None
    ultimo_registro = None
    saldo_total = None

    with pdfplumber.open(ruta_pdf) as pdf:
        # 1. Columnas finales
        # Nombres en el archivo
        columnas = ["Fecha", "Comprobante", "Movimiento", "Débito", "Crédito", "Saldo en cuenta"]

        # 2. Patron de fechas con regex
        patron_fecha = re.compile(r"\b\d{2}/\d{2}/\d{2}\b") # .compile prepara un patrón de búsqueda.

        # 3. Distintos patrones de montos
        # Por ejemplo: $ 24.543,75 // -$ 3.051.185,33 // $ 33,87 // $ 800.000,00
        patron_monto = re.compile(r"-?\$\s?"r"\d{1,3}"r"(?:\.\d{3})*"r",\d{2}") 

        # 4. Convertir monto de texto a numero 
        # porque en Arg por ejemplo los decimales vienen con , y necestamos que sean con .
        # si no hay valor fila-fila se pone NA. Tambien considera los numeros negativos
        def convertir_monto(texto):
            if texto is None:
                return "NA"
            texto = texto.strip()
            negativo = "-" in texto
            texto = (texto.replace("$", "").replace("-", "").replace(" ", "").replace(".", "").replace(",", "."))
            try:
                valor = float(texto)
            except ValueError:
                return "NA"
            if negativo:
                valor *= -1
            return valor

        # 5. Configuracion de la fecha en el inicio de la linea
        # Hay veces que aparecen fechas en el concepto pero eso no lo tenemos que tner en cuenta
        def fecha_al_inicio(texto):
            resultado = re.match(r"^\s*(\d{2}/\d{2}/\d{2})", texto)
            if resultado:
                return resultado.group(1)
            return None


        # 6. Agrupar palabras en linea

        def agrupar_palabras_en_lineas(palabras, tolerancia_vertical = 3):
            palabras = sorted(palabras, key = lambda palabra: palabra["top"])
            lineas = []
            for palabra in palabras:
                agregada = False
                for linea in lineas:
                    top_linea = linea[0]["top"]
                    if (abs(palabra["top"] - top_linea) <= tolerancia_vertical):
                        linea.append(palabra)
                        agregada = True
                        break
                if not agregada:
                    lineas.append([palabra])
            # Ordenar horizontalmente
            for linea in lineas:
                linea.sort(key=lambda palabra: palabra["x0"])
            return lineas

        # 7. Extraer solo fecha
        def es_fecha_sola(texto):
            return bool(re.fullmatch(r"\d{2}/\d{2}/\d{2}", texto.strip()))

        # 8. Setear el numero de pagina 
        def es_numero_de_pagina(texto):
            return bool(re.fullmatch(r"\d+\s*-\s*\d+", texto.strip()))

        # 9. Informacion que no interesa 
        #Otras para agregar?? es la info que esta al final o al principio del pdf
        def es_informacion_a_ignorar(texto):
            texto_lower = texto.lower()
            # Informacion de la cta bancaria
            if "cuenta corriente nº" in texto_lower:
                return True
            if "cuenta corriente nro" in texto_lower:
                return True
            if "cbu:" in texto_lower:
                return True
            # Informacion del banco
            if ("banco santander argentina s.a." in texto_lower):
                return True
            if ("correlativo 800678" in texto_lower):
                return True 
            if ("ningún accionista mayoritario" in texto_lower):
                return True
            if ("tampoco lo hacen otras entidades" in texto_lower):
                return True
            if ("salvo error u omisión" in texto_lower):
                return True
            
            # Informacion del acuerdo bancario
            if texto_lower == "acuerdo":
                return True
            if texto_lower.startswith("límite:"):
                return True
            if texto_lower.startswith("limite:"):
                return True
            if "vencimiento:" in texto_lower:
                return True
            if "total numerales:" in texto_lower:
                return True
            if "total excedido:" in texto_lower:
                return True
            if "máximo saldo deudor:" in texto_lower:
                return True
            if "maximo saldo deudor:" in texto_lower:
                return True

            # Encabezado de la tabla 
            if ("fecha" in texto_lower  and "comprobante" in texto_lower and "movimiento" in texto_lower):
                return True
            return False

        # 10. Lineas explicativas
        def es_linea_explicativa(texto):
            texto_lower = (texto.lower().strip())
            patrones = ["resp:", "a ", "de ", "del ", "comision transferencias", "comisión transferencias", "lx argentina",
                "federacion patro", "federación patro", "bbva consolidar", "sancor cooperati", "camino de las si"]
            for patron in patrones:
                if texto_lower.startswith(patron):
                    return True
            return False

        # 11. Limpiar info del movimiento

        def limpiar_movimiento(texto):
            # Eliminar solamente una fecha
            # si aparece al inicio del texto
            texto = re.sub(r"^\s*\d{2}/\d{2}/\d{2}\s*", "", texto)
            # Eliminar espacios repetidos
            texto = re.sub(r"\s+", " ", texto)
            return texto.strip()

        # 12. Extraer el numero de comprobante y el movimiento
        def extraer_comprobante_y_movimiento(texto):
            # Eliminar fecha solamente al principio
            texto = re.sub(r"^\s*\d{2}/\d{2}/\d{2}\s*", "", texto)

            # Eliminar montos
            texto = patron_monto.sub("", texto)

            # Limpiar espacios
            texto = re.sub(r"\s+"," ", texto).strip()

            # Separar comprobante
            partes = texto.split()
            comprobante = "NA"
            if (len(partes) > 0 and partes[0].isdigit()):
                comprobante = partes[0]
                partes = partes[1:]
            movimiento = " ".join(partes)
            movimiento = limpiar_movimiento(movimiento)
            return comprobante, movimiento

        # 13. Variables principales
        registros = []
        leyendo_movimientos = False
        fin_movimientos = False
        fecha_actual = None
        ultimo_registro = None
        # Variable para guardar el saldo total
        # que aparece en la ultima línea despues del ultimo valor de la tabla
        saldo_total = None

        # 14. Recorrer todo el pdf
        with pdfplumber.open(ruta_pdf) as pdf:
            print(f"Cantidad de páginas: " f"{len(pdf.pages)}")
            for numero_pagina, pagina in enumerate(pdf.pages, start=1):
                # Si terminó la tabla, no procesar más páginas
                if fin_movimientos:
                    break
                print(f"Procesando página " f"{numero_pagina}...")

                # Extraer palabras
                palabras = pagina.extract_words()
                # Agrupar palabras en líneas
                lineas = agrupar_palabras_en_lineas(palabras)

                # Procesar cada línea
                for linea in lineas:
                    texto = " ".join(palabra["text"] for palabra in linea)
                    texto = texto.strip()
                    # Inicio de movimientos
                    if (texto.lower() == "movimientos"):
                        leyendo_movimientos = True
                        print("Inicio de movimientos")
                        continue
                    # Fin de procemiento por saldo total
                    #Esto se agrego despues porque necesaitaba este valor para hacer una comprobacion de que este todo ok, debitos +creditos = saldo
                    if (leyendo_movimientos and "saldo total" in texto.lower()):
                        print( "Fin movimientos: " "Saldo total encontrado")

                        # Extraer el monto del saldo total
                        montos_saldo_total = (patron_monto.findall(texto))
                        if len(montos_saldo_total) > 0: 
                            saldo_total = convertir_monto(montos_saldo_total[-1])
                        fin_movimientos = True
                        leyendo_movimientos = False
                        break #Salir

                    # Fin por detalle impositivo, que seria la seccion que viene despues
                    # Esto es porque se conoce el pdf bancario
                    if (leyendo_movimientos and "detalle impositivo" in texto.lower()):
                        print("Fin de movimientos")
                        fin_movimientos = True
                        leyendo_movimientos = False
                        break

                    # Si no estamoos en movimientos
                    # Creo que esta parte se repite porque ya se define antes, revisar!
                    if not leyendo_movimientos:
                        continue

                    # Ignorar informacion general delp pdf
                    if es_informacion_a_ignorar(texto):
                        continue
                    # Ignorar el numero de pagina
                    if es_numero_de_pagina(texto):
                        continue

                    # Actualizar la info de la fecha             #
                    # SOLO si está al principio, esto evita que fechas internas como "Del 01/04/26 al 30/04/26"
                    # cambien la fecha del movimiento.
                    fecha_en_linea = fecha_al_inicio(texto)
                    if fecha_en_linea:
                        fecha_actual = fecha_en_linea

                    # Extraer montos
                    montos = patron_monto.findall(texto)
                    
                    # Lineas explicativas importantes 
                    if es_linea_explicativa(texto):
                        if ultimo_registro is not None:
                            texto_limpio = limpiar_movimiento(texto)
                            if texto_limpio:
                                ultimo_registro["Movimiento"] += (" " + texto_limpio)
                        continue

                    # Lineas sin montos 
                    if len(montos) == 0:
                        # Fecha sola
                        if es_fecha_sola(texto):
                            continue
                        # Texto explicativo
                        if ultimo_registro is not None:
                            texto_limpio = limpiar_movimiento(texto)
                            if texto_limpio:
                                ultimo_registro["Movimiento"] += (" " + texto_limpio)
                        continue

                    # Ultimo monto = saldo, esto se repite creo? revisar
                    saldo = convertir_monto(montos[-1])

                # El penunltimo valor corresponde al Importe final del mes 
                    if len(montos) >= 2:
                        importe = convertir_monto(montos[-2])
                    else:
                        importe = "NA"

                    # Extraer comprobante y movimiento

                    texto_sin_montos = patron_monto.sub("", texto)
                    comprobante, movimiento = (extraer_comprobante_y_movimiento(texto_sin_montos))

                    # Hay que ver, clasificar si corresponde a valor de debito o credito
                    #Una forma: Se compara el saldo actual con el saldo anterior.
                    # # Si el saldo aumenta: Crédito
                    # Si el saldo disminuye: Débito
                    
                    #Otra forma: con los espacios pero no funciono porque el monto puede variar y la cantidad de espacios/caracteres puede variar
                    debito = "NA"
                    credito = "NA"
                    #Verifica que no este vacio, que sea entero o decimal, 
                    if (len(registros) > 0 and isinstance(saldo,(int, float)) and isinstance(registros[-1]["Saldo en cuenta"],(int, float))
                        and isinstance(importe,(int, float))):
                        saldo_anterior = (registros[-1]["Saldo en cuenta"])
                        # Saldo aument
                        if saldo > saldo_anterior:
                            credito = importe
                        # Saldo disminuye
                        elif saldo < saldo_anterior:
                            debito = importe
                    else:
                        # Primer movimiento: Saldo Inicial que no tiene débito ni crédito.
        
                        debito = "NA"
                        credito = "NA"

                    # Crear registro
                    registro = {"Fecha": fecha_actual, "Comprobante": comprobante,"Movimiento": movimiento, "Débito": debito,"Crédito": credito,
                        "Saldo en cuenta": saldo}
                    registros.append(registro)
                    ultimo_registro = registros[-1]


        # 15. Crear df
        df_movimientos = pd.DataFrame(registros, columns = columnas)

        # 16. Completar info vacio
        df_movimientos = (df_movimientos.fillna("NA"))

        # 17. Guardar saldo inicial
        saldo_inicial = (df_movimientos.iloc[0]["Saldo en cuenta"])

        # 18. Eliminar saldo inicial
        df_movimientos = (df_movimientos.iloc[1:].reset_index(drop=True))

        # 19. Limpieza final del movimiento de interes
        df_movimientos["Movimiento"] = (df_movimientos["Movimiento"].astype(str).str.replace(r"\s+"," ",regex=True).str.strip())
        #print(df_movimientos["Movimiento"] )

        # 20. Convertir debitos y creditos a numeros
        df_movimientos["Débito_num"] = pd.to_numeric(df_movimientos["Débito"],errors="coerce") # que es coerce? lo dejamos prque asi funciona
        df_movimientos["Crédito_num"] = pd.to_numeric(df_movimientos["Crédito"], errors="coerce")


        # 21. Calcular total de debitos
        total_debitos = (df_movimientos["Débito_num"].fillna(0).sum())
        #print(total_debitos)

        # 22. Calcular creditos
        total_creditos = (df_movimientos["Crédito_num"].fillna(0).sum())

        # 23. Calcular el saldo final para comprobar que todos los debitos/creditos estan bien ubicados
        # y que no hay valores de mas
        saldo_calculado = (saldo_inicial - total_debitos + total_creditos)


        # 24. CEvaluacion final si est todo ok
        # Lo dejamos pero no lo mostamos en el github 
        # print("\n")
        # print("=" * 100)
        # print("Comprobacion de saldo")
        # print("=" * 100)
        # print("Saldo inicial:", saldo_inicial)
        # print("Total débitos:", total_debitos)
        # print( "Total créditos:", total_creditos)
        # print("Saldo calculado:",saldo_calculado)
        # print("Saldo total del extracto:",saldo_total)
        # print("\n")

        # Comparar 
        if (round(saldo_calculado, 2) == round(saldo_total, 2)):
            print("Comprobacion correcta")
            print("El saldo inicial, los débitos y los créditos coinciden con el saldo total del extracto.")
        else: 
            diferencia = (saldo_calculado -saldo_total)
            print("Comprobacion incorrecta")
            print("Diferencia:", round(diferencia, 2))

        # 25. Eliminar algunas columnas que no son de interes
        df_movimientos = df_movimientos.drop(columns=["Débito_num","Crédito_num"])

        # 26. Mostrar info del df final
        # print("\n")
        # print("=" * 100)
        # print("DATAFRAME FINAL")
        # print("=" * 100)
        # print(df_movimientos)
        # print("\n")
        # print("Cantidad total de movimientos:", len(df_movimientos))


        # # 26. Exportar excel/csv
        # df_movimientos.to_excel("movimientos_bancarios.xlsx", index=False)
        #df_movimientos.to_csv("movimientos_bancarios.csv")
    return df_movimientos


In [ ]:
pdf = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/resumenes_bancarios/Banco_05-2026.pdf"
extraer_movimientos_bancarios(pdf)

Cantidad de páginas: 8
Procesando página 1...
Inicio de movimientos
Procesando página 2...
Procesando página 3...
Procesando página 4...
Procesando página 5...
Fin movimientos: Saldo total encontrado
Comprobacion correcta
El saldo inicial, los débitos y los créditos coinciden con el saldo total del extracto.


#### Recorriendo una carpeta con resumenes bancarios

In [ ]:
carpeta = "D:/Josefina/Proyectos/Datascience/Data_Analytics/Contabilidad_Empresa/data/raw/resumenes_bancarios"
archivos = []
os.listdir(carpeta)
for archivo in os.listdir(carpeta):

    # Verificar que sea un PDF
    if archivo.lower().endswith(".pdf"):
        print(archivo)

        # Crear la ruta completa
        ruta_pdf = os.path.join(carpeta, archivo)

        # Aplicar tu función
        df = extraer_movimientos_bancarios(ruta_pdf)
         # Agregar el nombre del archivo como una nueva columna
        df["archivo_origen"] = os.path.splitext(archivo)[0]
        # Guardar el DataFrame en la lista
        archivos.append(df)
        # Unir todos los DataFrames
    df_final = pd.concat(archivos, ignore_index=True)

print(df_final)

In [84]:
# llo guardamos para visualizarlo
df_final.to_csv("movimientos_bancarios.csv")